# Find quality=some entities from active non-government providers

**Author**: Sian Teesdale  
**Date created**: 2nd July 2026  
**Dataset Scope**: datasets with an active local-authority/national-park-authority/development-corporation provider  
**Purpose**: Find entities where `quality = some`, provided by an organisation that (a) currently has an active endpoint — i.e. is actively submitting data, via `source`/`source_pipeline`, same as `reports/count_organisations_providers_platform/run_analysis.py` — and (b) is not a central `government-organisation:` (e.g. MHCLG, Historic England), but rather a `local-authority:`, `national-park-authority:`, or `development-corporation:`. Follows on from [digital-land/config#2651](https://github.com/digital-land/config/issues/2651).

This is a **provider-first** approach: identify the active non-government providers first (from `source`/`source_pipeline`), then check the quality of the entities they've submitted (via `organisation_entity` on the dataset's entity table) — rather than starting from entity ownership and filtering down after.

In [16]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd

from helpers import fetch_filtered_table_csv, fetch_table_csv

# Org prefixes that count as "not a government organisation" for this ticket.
NON_GOV_PREFIXES = ("local-authority:", "national-park-authority:", "development-corporation:")

# Notebook's cwd is its own directory (analysis/2026-07_.../) by default in Jupyter.
DATA_DIR = os.path.join("..", "..", "data")
os.makedirs(DATA_DIR, exist_ok=True)

## 1. Organisation lookup

Build `organisation_entity` → organisation curie / name maps from the `digital-land` Datasette `organisation` table.

In [17]:
print("Fetching organisation table...")
org_df = fetch_table_csv("organisation")

entity_to_org = org_df.set_index("entity")["organisation"].to_dict()
org_to_name = org_df.set_index("organisation")["name"].to_dict()

print(f"  {len(org_df)} organisations loaded")
org_df[["organisation", "name", "entity"]].head()

Fetching organisation table...
  545 organisations loaded


,organisation,name,entity
0,development-corporation:Q105544651,Aycliffe and Peterlee Development Corporation,600003
1,development-corporation:Q105544654,Basildon Development Corporation,600004
2,development-corporation:Q105544669,Telford Development Corporation,600007
3,development-corporation:Q115585981,Stockport Town Centre West Mayoral Development...,4700006
4,development-corporation:Q117149370,Middlesbrough Development Corporation,4700001


## 2. Active non-government providers

Join `source` + `source_pipeline` (same as `run_analysis.py`) to find (dataset, organisation) pairs with a currently active endpoint — i.e. the organisation is actually providing data right now, not just historically attributed via an entity range. Filter to organisations with prefix `local-authority:`, `national-park-authority:`, or `development-corporation:`.

In [18]:
print("Fetching source and source_pipeline...")
source = fetch_table_csv("source")
source_pipeline = fetch_table_csv("source_pipeline")
merged_source = source.merge(source_pipeline, on="source")
active_source = merged_source[merged_source["endpoint"] != ""]

active_providers = (
    active_source[["pipeline", "organisation"]]
    .drop_duplicates()
    .rename(columns={"pipeline": "dataset"})
)
active_providers["org_prefix"] = active_providers["organisation"].str.split(":").str[0] + ":"

non_gov_providers = active_providers[active_providers["org_prefix"].isin(NON_GOV_PREFIXES)].copy()
non_gov_providers["organisation_name"] = non_gov_providers["organisation"].map(org_to_name)

target_datasets = sorted(non_gov_providers["dataset"].unique())
print(f"{len(non_gov_providers)} active non-government (dataset, organisation) providers "
      f"across {len(target_datasets)} datasets")
non_gov_providers.head()

Fetching source and source_pipeline...
3577 active non-government (dataset, organisation) providers across 29 datasets


,dataset,organisation,org_prefix,organisation_name
0,brownfield-land,local-authority:COP,local-authority:,Copeland Borough Council
1,developer-agreement,local-authority:TAM,local-authority:,Tameside Metropolitan Borough Council
2,developer-agreement-contribution,local-authority:TAM,local-authority:,Tameside Metropolitan Borough Council
3,developer-agreement-transaction,local-authority:TAM,local-authority:,Tameside Metropolitan Borough Council
4,conservation-area,local-authority:CHA,local-authority:,Charnwood Borough Council


## 3. Query `quality = some` entities for those datasets

For each dataset that has at least one qualifying provider, stream its own Datasette entity table filtered to `quality = some` via the table CSV view (`_stream=on` + `?quality=some`) — one request per dataset, no SQL pagination. Only scans the datasets identified above, not the full ~280-dataset platform, so this is quick.

Run in parallel across datasets; datasets with no entity Datasette DB are skipped.

In [19]:
def _some_quality_for_dataset(dataset):
    try:
        df = fetch_filtered_table_csv(
            dataset,
            "entity",
            columns=["name", "reference", "organisation_entity", "quality"],
            quality="some",
        )
    except Exception:
        return None  # no Datasette entity table for this dataset (or a repeated server error)
    if df.empty:
        return None
    df["dataset"] = dataset
    return df


print(f"Querying quality='some' entities across {len(target_datasets)} datasets (parallel)...")
some_quality_parts = []
with ThreadPoolExecutor(max_workers=30) as pool:
    futures = {pool.submit(_some_quality_for_dataset, ds): ds for ds in target_datasets}
    for f in as_completed(futures):
        result = f.result()
        if result is not None:
            print(f"  {futures[f]}: {len(result)} quality=some entities")
            some_quality_parts.append(result)

some_quality_df = pd.concat(some_quality_parts, ignore_index=True) if some_quality_parts else pd.DataFrame()
print(f"\nTotal: {len(some_quality_df)} quality=some entities across {some_quality_df['dataset'].nunique()} datasets")

Querying quality='some' entities across 29 datasets (parallel)...
  article-4-direction: 2 quality=some entities
  central-activities-zone: 10 quality=some entities
  developer-agreement: 223 quality=some entities
  developer-agreement-transaction: 316 quality=some entities
  developer-agreement-contribution: 202 quality=some entities
  design-code: 11 quality=some entities
  brownfield-land: 751 quality=some entities
  listed-building-outline: 232 quality=some entities
  infrastructure-funding-statement: 205 quality=some entities
  minerals-plan: 65 quality=some entities
  development-corporation-boundary: 9 quality=some entities
  waste-plan: 71 quality=some entities
  conservation-area: 5872 quality=some entities
  brownfield-site: 2124 quality=some entities
  local-plan: 911 quality=some entities
  conservation-area-document: 10398 quality=some entities
  plan-timetable: 8137 quality=some entities
  listed-building: 380020 quality=some entities

Total: 409559 quality=some entities 

## 4. Keep entities provided by an active non-government provider

Map each entity's `organisation_entity` to its organisation curie, then inner-join against the active non-government providers list from step 2 — this applies both the "active" and "non-government" filters in one step.

In [20]:
some_quality_df["organisation_entity"] = pd.to_numeric(
    some_quality_df["organisation_entity"], errors="coerce"
)
some_quality_df = some_quality_df.dropna(subset=["organisation_entity"]).copy()
some_quality_df["organisation_entity"] = some_quality_df["organisation_entity"].astype(int)
some_quality_df["organisation"] = some_quality_df["organisation_entity"].map(entity_to_org)

flagged_df = some_quality_df.merge(
    non_gov_providers[["dataset", "organisation", "organisation_name"]],
    on=["dataset", "organisation"],
    how="inner",
)
flagged_df["entity_url"] = "https://www.planning.data.gov.uk/entity/" + flagged_df["entity"].astype(str)

flagged_df = flagged_df[
    ["dataset", "entity", "name", "reference", "organisation", "organisation_name", "quality", "entity_url"]
].sort_values(["dataset", "organisation", "entity"])

print(f"{len(flagged_df)} entities, quality=some, from an active non-government provider, "
      f"across {flagged_df['dataset'].nunique()} datasets and {flagged_df['organisation'].nunique()} organisations")
flagged_df.head(20)

7492 entities, quality=some, from an active non-government provider, across 13 datasets and 242 organisations


,dataset,entity,name,reference,organisation,organisation_name,quality,entity_url
0,article-4-direction,6102345,Newnham (within conservation area),ART4/001,local-authority:FEN,Fenland District Council,some,https://www.planning.data.gov.uk/entity/6102345
1,article-4-direction,6102346,"English Bicknor (land at Redhouse Lane, Murrel...",ART4/002,local-authority:FEN,Fenland District Council,some,https://www.planning.data.gov.uk/entity/6102346
159,brownfield-land,1716900,B/02471/11,B/02471/11,local-authority:BNE,London Borough of Barnet,some,https://www.planning.data.gov.uk/entity/1716900
160,brownfield-land,1716941,F/03933/13,F/03933/13,local-authority:BNE,London Borough of Barnet,some,https://www.planning.data.gov.uk/entity/1716941
263,brownfield-land,1740606,766,766,local-authority:CHW,Cheshire West and Chester Council,some,https://www.planning.data.gov.uk/entity/1740606
264,brownfield-land,1740626,7430,7430,local-authority:CHW,Cheshire West and Chester Council,some,https://www.planning.data.gov.uk/entity/1740626
265,brownfield-land,1740627,7481,7481,local-authority:CHW,Cheshire West and Chester Council,some,https://www.planning.data.gov.uk/entity/1740627
266,brownfield-land,1740628,7490,7490,local-authority:CHW,Cheshire West and Chester Council,some,https://www.planning.data.gov.uk/entity/1740628
267,brownfield-land,1740629,7500,7500,local-authority:CHW,Cheshire West and Chester Council,some,https://www.planning.data.gov.uk/entity/1740629
271,brownfield-land,1741876,6889,6889,local-authority:CHW,Cheshire West and Chester Council,some,https://www.planning.data.gov.uk/entity/1741876


## 5. Explore — summary counts

In [21]:
by_dataset = (
    flagged_df.groupby("dataset")
    .agg(flagged_entities=("entity", "nunique"), organisations=("organisation", "nunique"))
    .sort_values("flagged_entities", ascending=False)
)
by_dataset

,flagged_entities,organisations
dataset,,
conservation-area,4687,199
brownfield-site,2124,27
plan-timetable,281,48
brownfield-land,116,8
local-plan,86,43
developer-agreement-transaction,69,1
developer-agreement,61,6
infrastructure-funding-statement,37,31
developer-agreement-contribution,26,1


In [23]:
by_org = (
    flagged_df.groupby(["organisation", "organisation_name"])
    .agg(flagged_entities=("entity", "nunique"), datasets=("dataset", "nunique"))
    .sort_values("flagged_entities", ascending=False)
)
by_org

,,flagged_entities,datasets
organisation,organisation_name,,
local-authority:WND,London Borough of Wandsworth,244,3
local-authority:WNUA,West Northamptonshire Council,217,4
local-authority:RDB,London Borough of Redbridge,203,2
local-authority:HRY,London Borough of Haringey,184,2
local-authority:ISL,London Borough of Islington,142,2
...,...,...,...
local-authority:TRF,Trafford Metropolitan Borough Council,1,1
local-authority:NLN,North Lincolnshire Council,1,1
local-authority:NET,Newcastle City Council,1,1


## 6. Export

Saved to the repo's top-level `data/` directory (gitignored) — used as input to `2_range_lookup_investigation.ipynb`.

In [24]:
out_path = os.path.join(DATA_DIR, "flagged_entities.csv")
flagged_df.to_csv(out_path, index=False)
print(f"Saved {len(flagged_df)} rows to {out_path}")

Saved 7492 rows to ../../data/flagged_entities.csv
